# Auditable Data Analysis Agent

**Author: Feras Elghandour**

Route natural-language data questions to explicit tools and keep the executed steps, cache behavior, and errors inspectable.

A deterministic tool router with dataset statistics, correlation analysis, bounded arithmetic parsing, working memory, and an in-memory audit trail.

Read README.md for result scope and DATA.md for input preparation. Run cells in order in a clean environment.

In [ ]:
import json
import re
import math
import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe
from collections import defaultdict

print(f"NumPy:   {np.__version__}")
print(f"Pandas:  {pd.__version__}")
print("Environment ready.")

In [ ]:
# Load the abalone dataset (external data source — tool will query this)
DF = pd.read_csv('abalone.csv')
DF_COLS_LOWER = {c.lower(): c for c in DF.columns}  # case-insensitive lookup map

print(f"Loaded: abalone.csv")
print(f"Shape:  {DF.shape[0]} rows × {DF.shape[1]} columns")
print(f"Columns: {DF.columns.tolist()}")
print(f"Sex distribution: {DF['Sex'].value_counts().to_dict()}")
DF.head(4)

In [ ]:
TOOL_REGISTRY = {}

def register_tool(fn):
    """Decorator: register function in the agent's tool registry."""
    TOOL_REGISTRY[fn.__name__] = fn
    return fn


@register_tool
def calculate(expression: str) -> str:
    """Evaluate a bounded arithmetic AST; attribute access and Python eval are absent."""
    import ast
    import operator
    binary = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
              ast.Div: operator.truediv, ast.Pow: operator.pow}
    functions = {'sqrt': math.sqrt, 'log': math.log, 'abs': abs, 'round': round}
    def visit(node):
        if isinstance(node, ast.Constant) and type(node.value) in (int, float):
            value = node.value
        elif isinstance(node, ast.Name) and node.id == 'pi':
            value = math.pi
        elif isinstance(node, ast.UnaryOp) and isinstance(node.op, (ast.UAdd, ast.USub)):
            value = visit(node.operand) * (-1 if isinstance(node.op, ast.USub) else 1)
        elif isinstance(node, ast.BinOp) and type(node.op) in binary:
            left, right = visit(node.left), visit(node.right)
            if isinstance(node.op, ast.Pow) and abs(right) > 12:
                raise ValueError('exponent exceeds supported limit')
            value = binary[type(node.op)](left, right)
        elif isinstance(node, ast.Call) and isinstance(node.func, ast.Name) and node.func.id in functions and not node.keywords:
            if not 1 <= len(node.args) <= 2:
                raise ValueError('unsupported argument count')
            value = functions[node.func.id](*(visit(arg) for arg in node.args))
        else:
            raise ValueError('unsupported expression')
        if type(value) not in (int, float) or not math.isfinite(value) or abs(value) > 1e12:
            raise ValueError('result outside supported range')
        return value
    try:
        if len(expression) > 200:
            raise ValueError('expression too long')
        tree = ast.parse(expression.replace('^', '**').strip(), mode='eval')
        if sum(1 for _ in ast.walk(tree)) > 64:
            raise ValueError('expression too complex')
        return f'Result: {round(visit(tree.body), 6)}'
    except (ValueError, TypeError, SyntaxError, OverflowError, ZeroDivisionError) as error:
        return f'Error evaluating expression: {error}'


@register_tool
def query_dataset(question: str) -> str:
    """
    Answer natural language questions about the abalone dataset.
    Supports: shape, columns, describe, sex distribution, per-sex statistics,
    per-column statistics (mean, std, min, max, median, count).
    """
    q = question.lower().strip()
    try:
        if any(k in q for k in ['shape', 'size', 'rows', 'how many row', 'how many sample']):
            return f"Dataset shape: {DF.shape[0]} rows × {DF.shape[1]} columns"
        if 'column' in q and 'correlat' not in q:
            return f"Columns: {DF.columns.tolist()}"
        if any(k in q for k in ['describe', 'summary statistics', 'descriptive']):
            return "Descriptive statistics:\n" + DF.describe().round(4).to_string()
        if any(k in q for k in ['sex distribution', 'sex count', 'class distribution',
                                  'how many male', 'how many female', 'how many infant']):
            counts = DF['Sex'].value_counts().to_dict()
            return (f"Sex distribution — Male (M): {counts.get('M',0)}, "
                    f"Female (F): {counts.get('F',0)}, Infant (I): {counts.get('I',0)}")
        # Sex-specific queries
        for sex_word, sex_code in [('female', 'F'), ('male', 'M'), ('infant', 'I')]:
            if sex_word in q:
                col = None
                for cname_l, cname in sorted(DF_COLS_LOWER.items(), key=lambda pair: -len(pair[0])):
                    if cname_l.replace('_', ' ') in q.replace('_', ' ') and cname_l != 'sex':
                        col = cname; break
                if col is None: col = 'Rings'
                sub = DF[DF['Sex'] == sex_code]
                stat = next((s for s in ['mean','std','min','max','median'] if s in q), 'mean')
                val = round(float(getattr(sub[col], stat)()), 4)
                return f"{stat.capitalize()} {col} for {sex_word}: {val}  (n={len(sub)})"
        # General column stat
        for cname_l, cname in sorted(DF_COLS_LOWER.items(), key=lambda pair: -len(pair[0])):
            if cname_l.replace('_', ' ') in q.replace('_', ' '):
                stat = next((s for s in ['mean','std','min','max','median','count'] if s in q), 'mean')
                val = round(float(getattr(DF[cname], stat)()), 4) if stat != 'count' else int(DF[cname].count())
                return f"{stat.capitalize()} of {cname}: {val}"
        return f"Could not parse query. Available columns: {DF.columns.tolist()}"
    except Exception as e:
        return f"Dataset query error: {e}"


@register_tool
def get_current_date(dummy: str = '') -> str:
    """Return the current date and time (UTC). No arguments needed."""
    return f"Current date/time: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}"


@register_tool
def correlation_analysis(col_pair: str) -> str:
    """
    Compute Pearson correlation between two dataset columns.
    Input format: 'column_name_1,column_name_2'  (case-insensitive)
    """
    try:
        parts = [p.strip() for p in col_pair.split(',')]
        if len(parts) != 2:
            return "Provide exactly two column names separated by a comma."
        resolved = []
        for p in parts:
            m = DF_COLS_LOWER.get(p.lower())
            if m: resolved.append(m)
            else:
                hits = [v for k, v in DF_COLS_LOWER.items() if p.lower() in k]
                resolved.append(hits[0] if hits else p)
        c1, c2 = resolved
        if c1 not in DF.columns or c2 not in DF.columns:
            return f"Column(s) not found. Available: {DF.columns.tolist()}"
        r = round(float(DF[c1].corr(DF[c2])), 4)
        strength = 'strong' if abs(r) > 0.7 else ('moderate' if abs(r) > 0.4 else 'weak')
        direction = 'positive' if r > 0 else 'negative'
        return f"Pearson r({c1}, {c2}) = {r}  [{strength} {direction} relationship]"
    except Exception as e:
        return f"Correlation error: {e}"


@register_tool
def check_scope(query: str) -> str:
    """
    Safeguard: verify whether the query falls within the agent's defined scope.
    Returns 'OUT_OF_SCOPE' if the query matches a blocked category.
    """
    blocked = ['password', 'personal data', 'credit card', 'medical diagnosis',
               'legal advice', 'stock', 'trading', 'financial advice', 'hack',
               'exploit', 'diagnose']
    if any(term in query.lower() for term in blocked):
        return 'OUT_OF_SCOPE'
    return 'IN_SCOPE'


print(f"Registered {len(TOOL_REGISTRY)} tools: {list(TOOL_REGISTRY.keys())}")

In [ ]:
class AgentMemory:
    """
    Two-level memory system for the ERL Research Agent.

    - conversation_history: sliding window of recent user/assistant turns
      (episodic short-term memory, max 10 exchanges)
    - working_memory: key-value store for caching tool results
      (semantic working memory, persists within session)
    """
    def __init__(self, max_history: int = 10):
        self.conversation_history = []
        self.working_memory = {}      # {cache_key: result}
        self.max_history = max_history

    def add_turn(self, role: str, content: str):
        """Append a conversation turn; prune if over max_history."""
        self.conversation_history.append({'role': role, 'content': content})
        if len(self.conversation_history) > self.max_history * 2:
            self.conversation_history = self.conversation_history[-self.max_history * 2:]

    def store(self, key: str, value: str):
        """Cache a tool result."""
        self.working_memory[key] = value

    def recall(self, key: str):
        """Retrieve cached result, or None if not found."""
        return self.working_memory.get(key)

    def summary(self):
        return (f"Conversation turns: {len(self.conversation_history)}  |  "
                f"Cached results: {len(self.working_memory)}")


# Quick test
mem = AgentMemory()
mem.add_turn('user', 'Hello')
mem.store('test_key', 'test_value')
print("Memory test:", mem.summary())
print("Recall test:", mem.recall('test_key'))

In [ ]:
SCOPE_BLOCKLIST = [
    'password', 'credit card', 'medical diagnosis', 'legal advice',
    'stock tip', 'diagnose', 'exploit', 'hack'
]

def classify_intent(query: str) -> tuple:
    """
    Map a natural language query to (tool_name, tool_input).

    Decision priority:
    1. Scope check (safety-first)
    2. Correlation keywords
    3. Numeric/calculation patterns
    4. Date/time keywords
    5. Default → dataset query
    """
    q = query.lower()

    # Priority 1: Scope guard
    if any(t in q for t in SCOPE_BLOCKLIST):
        return 'check_scope', query

    # Priority 2: Correlation
    if 'correlat' in q:
        cols = re.findall(
            r'\b(length|diameter|height|whole_weight|shucked_weight|'
            r'viscera_weight|shell_weight|rings)\b', q
        )
        if len(cols) >= 2:
            return 'correlation_analysis', f'{cols[0]},{cols[1]}'
        return 'correlation_analysis', 'rings,shell_weight'

    # Priority 3: Calculation
    if re.search(r'\d.*[\+\-\*\/]|sqrt\(|square root', q):
        m = re.search(r'square root of ([\d.]+)', q)
        if m:
            return 'calculate', f'sqrt({m.group(1)})'
        expr = re.sub(r'(calculate|compute|what is|=)', '', query, flags=re.IGNORECASE).strip()
        expr = re.sub(r'[^0-9\+\-\*\/\(\)\.\s\^sqrtpiabsroundlog]', '', expr).strip()
        return 'calculate', expr or query

    # Priority 4: Date/time
    if any(k in q for k in ['date', 'time', 'today', 'current time']):
        return 'get_current_date', ''

    # Priority 5: Dataset query (default)
    return 'query_dataset', query


# Test classifier
test_q = [
    "How many rows are in the dataset?",
    "What is the correlation between Rings and Shell_weight?",
    "Calculate the square root of 4177",
    "What is today's date?",
    "Can you diagnose my illness?",
]
print("Intent classification tests:")
for q in test_q:
    tool, inp = classify_intent(q)
    print(f"  Q: '{q[:50]}'")
    print(f"     → tool='{tool}', input='{inp[:40]}'")

In [ ]:
class ResearchAgent:
    """
    ERL Data Research Agent — ReAct pattern implementation.

    Implements the Thought → Action → Observation loop (Yao et al., 2023)
    with memory, tool registry, scope guards, and an immutable audit log.

    Safeguards:
        1. Scope guard: blocks out-of-scope queries before tool dispatch
        2. Step limiter: prevents infinite reasoning loops (max_steps)
        3. Tool sandboxing: calculate() uses restricted eval()
        4. Working memory cache: prevents redundant external calls
        5. Append-only audit log: full observability of every decision
    """
    def __init__(self, name: str = 'ERL-RA-v1', max_steps: int = 5):
        self.name = name
        self.max_steps = max_steps
        self.memory = AgentMemory(max_history=10)
        self.audit_log = []         # append-only, never modified after write

    def _log(self, event: str, content: str):
        """Append a timestamped entry to the audit log."""
        self.audit_log.append({
            'ts':      datetime.datetime.now().strftime('%H:%M:%S.%f')[:12],
            'agent':   self.name,
            'event':   event,
            'content': str(content)[:250]
        })

    def run(self, query: str) -> dict:
        """
        Process a user query through the ReAct loop.

        Returns a result dict with keys:
            step, tool, tool_input, observation, response, cached
        """
        self.memory.add_turn('user', query)
        self._log('USER_INPUT', query)

        for step in range(1, self.max_steps + 1):

            # ── THINK: classify intent ──────────────────────────────────────
            tool_name, tool_input = classify_intent(query)
            thought = f"Step {step}: intent='{tool_name}', input='{tool_input[:60]}'"
            self._log('THINK', thought)

            # ── CHECK WORKING MEMORY ────────────────────────────────────────
            cache_key = f"{tool_name}|{tool_input[:80]}"
            cached_result = self.memory.recall(cache_key)

            # ── ACT: call tool or serve from cache ──────────────────────────
            if cached_result is not None:
                observation = cached_result
                self._log('CACHE_HIT', f'key={cache_key[:60]}')
            else:
                if tool_name not in TOOL_REGISTRY:
                    observation = f"Tool '{tool_name}' not registered."
                    self._log('ERROR', observation)
                else:
                    try:
                        observation = TOOL_REGISTRY[tool_name](tool_input)
                    except Exception as e:
                        observation = f"Tool execution error: {e}"
                        self._log('ERROR', str(e))
                self.memory.store(cache_key, observation)
                self._log('ACT', f"{tool_name} → {str(observation)[:100]}")

            # ── OBSERVE & SYNTHESIZE ────────────────────────────────────────
            if 'OUT_OF_SCOPE' in str(observation):
                response = (
                    "⚠️  That request is outside my defined scope. I am a research "
                    "data assistant for the abalone dataset. I can answer statistical "
                    "questions, run calculations, and analyze the data."
                )
                self._log('SAFEGUARD', 'out-of-scope query blocked')
            else:
                response = f"📊 {observation}"

            self.memory.add_turn('assistant', response)
            self._log('RESPONSE', response[:180])

            return {
                'step': step, 'tool': tool_name, 'tool_input': tool_input,
                'observation': str(observation), 'response': response,
                'cached': cached_result is not None
            }

        # Safeguard: step limit exceeded
        fallback = '⚠️  Reasoning step limit reached. Please simplify your query.'
        self._log('SAFEGUARD', 'max_steps exceeded')
        return {'step': self.max_steps, 'tool': 'none', 'response': fallback, 'cached': False}


# Instantiate agent
agent = ResearchAgent(name='ERL-RA-v1', max_steps=5)
print(f"Agent '{agent.name}' initialized.")
print(f"Tools: {list(TOOL_REGISTRY.keys())}")
print(f"Max steps: {agent.max_steps}")

In [ ]:
scenarios = [
    ('Basic dataset query',     'How many rows and columns are in the dataset?'),
    ('Sex-filtered statistic',  'What is the mean shell weight of female abalone?'),
    ('Correlation analysis',    'What is the correlation between Rings and Shell_weight?'),
    ('Calculation',             'Calculate the square root of 4177'),
    ('Cache hit (repeat)',      'What is the correlation between Rings and Shell_weight?'),
    ('Date tool',               'What is today\'s date?'),
    ('Out-of-scope (safeguard)','Can you give me a medical diagnosis for my symptoms?'),
    ('General column stat',     'What is the mean rings value for all abalone?'),
    ('Sex distribution',        'What is the sex distribution of the dataset?'),
    ('Descriptive stats',       'Give me descriptive statistics for the dataset'),
]

print(f"{'='*72}")
print(f"ERL DATA RESEARCH AGENT — SESSION LOG")
print(f"Agent: {agent.name}  |  Dataset: abalone.csv ({len(DF):,} samples)")
print(f"{'='*72}")

all_results = []
for label, query in scenarios:
    result = agent.run(query)
    result['label'] = label
    result['query'] = query
    all_results.append(result)
    cached_tag = ' [CACHE HIT]' if result.get('cached') else ''
    print(f"\n[{label}]")
    print(f"  USER : {query}")
    print(f"  TOOL : {result['tool']}{cached_tag}")
    print(f"  AGENT: {result['response'][:200]}")

print(f"\n{'='*72}")
print(f"Session complete. {len(all_results)} queries processed.")
print(agent.memory.summary())

In [ ]:
print("FULL AUDIT LOG:")
print(f"{'Time':>12}  {'Event':>16}  Content")
print('-' * 80)
for entry in agent.audit_log:
    print(f"  {entry['ts']:>12}  {entry['event']:>16}  {entry['content'][:55]}")

# Count events by type
from collections import Counter
event_counts = Counter(e['event'] for e in agent.audit_log)
print(f"\nEvent summary: {dict(event_counts)}")

In [ ]:
print("WORKING MEMORY CONTENTS:")
print(f"{'Key':>55}  Value (truncated)")
print('-' * 90)
for k, v in agent.memory.working_memory.items():
    print(f"  {k[:52]:>52}  {str(v)[:40]}")

In [ ]:
# Regression check for natural-language column normalization.
result = query_dataset('What is the mean shell weight of female abalone?')
expected = round(float(DF.loc[DF['Sex'] == 'F', 'Shell_weight'].mean()), 4)
assert 'Shell_weight' in result and str(expected) in result, result
assert calculate('sqrt(16) + 2^3') == 'Result: 12.0'
assert calculate("().__class__").startswith('Error')
assert calculate('9**99999').startswith('Error')
print('Column-resolution and bounded-calculation regression checks passed.')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# ── Left: ReAct Loop Diagram ──────────────────────────────────────────────────
ax = axes[0]
ax.set_xlim(0, 10); ax.set_ylim(0, 12); ax.axis('off')
ax.set_title('Figure 1a — ReAct Agent Loop', fontsize=11, fontweight='bold')

boxes = [
    (5, 11, 'User Query', '#4a90d9', 'white'),
    (5,  9, 'Scope Guard\n(Safeguard)', '#e74c3c', 'white'),
    (5,  7, 'Intent Classifier\n(THINK)', '#2ecc71', 'white'),
    (5,  5, 'Working Memory\nCache Check', '#f39c12', 'white'),
    (5,  3, 'Tool Registry\n(ACT)', '#9b59b6', 'white'),
    (5,  1, 'Response + Audit Log\n(OBSERVE)', '#1abc9c', 'white'),
]
for x, y, label, color, tc in boxes:
    ax.add_patch(plt.Rectangle((x-1.8, y-0.6), 3.6, 1.2,
                                facecolor=color, edgecolor='white', linewidth=1.5,
                                alpha=0.85, zorder=3))
    ax.text(x, y, label, ha='center', va='center', color=tc, fontsize=9,
            fontweight='bold', zorder=4)

for y_from, y_to in [(10.4,9.6),(8.4,7.6),(6.4,5.6),(4.4,3.6),(2.4,1.6)]:
    ax.annotate('', xy=(5, y_to), xytext=(5, y_from),
                arrowprops=dict(arrowstyle='->', color='#555', lw=1.5))

ax.annotate('', xy=(3.5, 9), xytext=(3.5, 1),
            arrowprops=dict(arrowstyle='->', color='#e74c3c', lw=1.5, linestyle='dashed'))
ax.text(2.8, 5, 'Block', color='#e74c3c', fontsize=8, rotation=90)

# ── Right: Event type breakdown ───────────────────────────────────────────────
ax2 = axes[1]
from collections import Counter
event_counts = Counter(e['event'] for e in agent.audit_log)
labels = list(event_counts.keys())
values = list(event_counts.values())
colors = ['#4a90d9','#2ecc71','#f39c12','#9b59b6','#1abc9c','#e74c3c','#95a5a6']
bars = ax2.barh(labels, values, color=colors[:len(labels)], alpha=0.85, edgecolor='white')
ax2.set_title('Figure 1b — Audit Log Event Counts', fontsize=11, fontweight='bold')
ax2.set_xlabel('Count')
for bar, val in zip(bars, values):
    ax2.text(val + 0.1, bar.get_y() + bar.get_height()/2, str(val), va='center', fontsize=9)
ax2.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('fig1_architecture.png', dpi=120, bbox_inches='tight')
plt.show()